In [ ]:
!pip -q install python-docx rank-bm25 requests python-dotenv

import os
import re
import copy
import requests
from docx import Document
from rank_bm25 import BM25Okapi
from google.colab import files
import getpass
from dotenv import load_dotenv

In [ ]:
uploaded = files.upload()

docx_files = [f for f in uploaded.keys() if f.lower().endswith(".docx")]
if not docx_files:
    raise FileNotFoundError("No DOCX uploaded. Please upload your template DOCX file.")

DOCX_PATH = docx_files[0]
print("Using DOCX:", DOCX_PATH)

ORIGINAL_DOC = Document(DOCX_PATH)
print("DOCX loaded.")

In [ ]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing from your .env file")

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
ROUTER_URL = "https://router.huggingface.co/v1/chat/completions"

def is_heading(line):
    line = (line or "").strip()
    return (
        len(line) >= 6
        and re.fullmatch(r"[A-Z0-9/ &\-\(\)\.,]+", line)
        and sum(c.isalpha() for c in line) >= 5
    )

def build_sections(doc):
    sections = []
    title = "UNTITLED"
    lines, idxs = [], []

    for i, p in enumerate(doc.paragraphs):
        t = (p.text or "").strip()

        if t == "":
            if lines and lines[-1] != "":
                lines.append("")
            if idxs:
                idxs.append(i)
            continue

        if is_heading(t) and (not lines or len(lines) > 3):
            if len("\n".join(lines).strip()) > 80:
                sections.append({"title": title, "text": "\n".join(lines), "idx": idxs})
            title, lines, idxs = t, [], []
        else:
            lines.append(p.text)
            idxs.append(i)

    if len("\n".join(lines).strip()) > 80:
        sections.append({"title": title, "text": "\n".join(lines), "idx": idxs})

    return sections

def tokenize(text):
    return re.findall(r"[a-zA-Z0-9']+", text.lower())

SECTIONS = build_sections(ORIGINAL_DOC)
BM25 = BM25Okapi([tokenize(s["title"] + " " + s["text"]) for s in SECTIONS])

def retrieve_top_k(email, k=5):
    scores = BM25.get_scores(tokenize(email))
    idxs = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [SECTIONS[i] for i in idxs]

def llm_choose(email, candidates):
    preview = []
    for i, c in enumerate(candidates, 1):
        preview.append(
            f"Option {i}:\nTitle: {c['title']}\n"
            + "\n".join(c["text"].splitlines()[:8])
        )

    prompt = f"""
Choose the template that best matches the email.
Reply ONLY with 1, 2, or 3.

Email:
{email}

Templates:
{"\n\n".join(preview)}
"""

    r = requests.post(
        ROUTER_URL,
        headers={
            "Authorization": f"Bearer {HF_TOKEN}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODEL_ID,
            "messages": [
                {"role": "system", "content": "Reply with only one digit."},
                {"role": "user", "content": prompt},
            ],
            "max_tokens": 5,
            "temperature": 0,
        },
        timeout=60,
    )

    if r.status_code != 200:
        raise RuntimeError(r.text)

    m = re.search(r"[123]", r.json()["choices"][0]["message"]["content"])
    return int(m.group()) - 1 if m else 0

def choose_template(email):
    top = retrieve_top_k(email)
    return top[llm_choose(email, top[:3])] if len(top) >= 3 else top[0]

def export_doc(chosen, out="chosen_template.docx"):
    doc = Document()
    body = doc._element.body
    for i in chosen["idx"]:
        body.append(copy.deepcopy(ORIGINAL_DOC.paragraphs[i]._p))
    doc.save(out)

In [ ]:
print("Paste the student email. Press ENTER on an empty line.\n")

lines = []
while True:
    line = input()
    if line == "":
        break
    lines.append(line)

EMAIL = "\n".join(lines).strip()

In [ ]:
CHOSEN = choose_template(EMAIL)

print("\nChosen Template Title:\n")
print(CHOSEN["title"])

print("\nChosen Template Text:\n")
print(CHOSEN["text"])

In [ ]:
export_doc(CHOSEN)
files.download("chosen_template.docx")